# 3. Treinamento

Treino de um modelo YOLO sobre os datasets processados. Os pesos e métricas são salvos em `runs/<experiment_id>_<timestamp>/`.

## Preparação

In [ ]:
import sys
sys.path.append("..")
from dotenv import load_dotenv
load_dotenv("../.env", override=True)

from src import config, datasets, train

## Configuração

In [ ]:
EXPERIMENT_ID = "yolo-direct-all-datasets"
WEIGHTS       = "yolo11m.pt"
DATASET_STAGE = "processed"

# datasets de treino — subconjunto ou todos os disponíveis
DATASETS = datasets.available()
# opções: ["crowdhuman", "inside-bus-view", "passenger-deakin", "passenger-detection-bus"]

TRAIN_CONFIG = {
    "epochs":  10,
    "batch":   16,
    "imgsz":   640,
    "workers": 4,
    # "device": 0,
    "amp":     True,  # automatic mixed precision
    # adicionar chaves de augmentação aqui para sobrescrever os padrões do YOLO:
    # "fliplr": 0.5, "mosaic": 1.0, "degrees": 10.0, "hsv_s": 0.7
}

## Datasets

In [ ]:
data_yaml = datasets.prepare(DATASETS, stage=DATASET_STAGE)

for name in DATASETS:
    for split in ["train", "valid", "test"]:
        img_dir = config.PROCESSED_DIR / name / split / "images"
        if img_dir.exists():
            n = len(list(img_dir.glob("*.*")))
            print(f"  {name}/{split}: {n} imagens")

## Modelo

In [ ]:
from ultralytics import YOLO
model = YOLO(WEIGHTS)

## Treinamento

In [ ]:
run_dir = train.run(EXPERIMENT_ID, model, data_yaml, TRAIN_CONFIG)
print("Run dir:", run_dir)